In [1]:
import pandas as pd
import json
from folium.plugins import HeatMap

# Linking hottest, coldest and wettest places into a [Database, places.db](places.db)
--> Primary and Foreign Key is rankings

In [2]:
file_path = "../../data/biomes_data/ecoregions_coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Flatten data
# .items function returns a sort of list (dict_items data type that is non viewable that gives a iterable but not printable list of key-value pairs as tuples, so at top level I get biome and associated dictionaries) 
coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]


In [3]:
filepath = '../../data/weather_data/coldest_places.csv'
df_coldest_places=pd.read_csv(filepath)


# Drop the 'max_temperature' column, as places are only ranked based on lowest temperature reached! (for ranking purposes)
df_coldest_places = df_coldest_places.drop(columns=['max_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_coldest_places = pd.merge(df_coldest_places, coordinates_df, on="region", how="left") # Left merge to retain df_coldest_places columns while adding coordinates for locations in top 100 where there is a match
df_coldest_places = df_coldest_places.rename(columns={'min_temperature': 'temperature'}) 
df_coldest_places.to_csv('../../data/weather_data/coldest_places.csv', index=False) # replace original CSV with updated one with coordinates!
print(df_coldest_places)

                                        region  temperature  max_rainfall  \
0                   Canadian Low Arctic tundra        -37.5           0.0   
1              Cherskii-Kolyma mountain tundra        -37.1           0.3   
2                Canadian Middle Arctic Tundra        -34.8           0.0   
3              Great Lakes Basin desert steppe        -34.4           0.0   
4             Mid-Canada Boreal Plains forests        -33.2           0.0   
..                                         ...          ...           ...   
95        Gulf of St. Lawrence lowland forests        -15.9           0.0   
96                         Enderby Land tundra        -15.7           0.2   
97   Suiphun-Khanka meadows and forest meadows        -15.7           0.0   
98  Eastern Himalayan alpine shrub and meadows        -15.6           0.0   
99                  Eastern Gobi desert steppe        -15.5           0.0   

     latitude   longitude  
0   65.756146  -99.131159  
1   65.830129  144.

In [4]:
filepath = '../../data/weather_data/hottest_places.csv'
df_hottest_places=pd.read_csv(filepath)


# Drop the 'min_temperature' column, only interested in max_temperature as that is basis of comparison
df_hottest_places = df_hottest_places.drop(columns=['min_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_hottest_places = pd.merge(df_hottest_places, coordinates_df, on="region", how="left")
df_hottest_places = df_hottest_places.rename(columns={'max_temperature': 'temperature'})
print(df_hottest_places)
df_hottest_places.to_csv('../../data/weather_data/hottest_places.csv', index=False)

                                     region  temperature  max_rainfall  \
0                            Simpson desert         45.6           0.0   
1                      Eyre and York mallee         44.8           1.0   
2                 Tirari-Sturt stony desert         43.7           0.1   
3        Eastern Australia mulga shrublands         43.0           0.0   
4                      Naracoorte woodlands         41.3           0.0   
..                                      ...          ...           ...   
95                   Bahia interior forests         32.7           0.0   
96      Northeast Congolian lowland forests         32.7           0.2   
97                      Eastern Arc forests         32.7           0.0   
98  South Arabian plains and plateau desert         32.7           0.0   
99            Magdalena-Urabá moist forests         32.6           1.4   

     latitude   longitude  
0  -26.274999  139.385269  
1  -33.488323  135.974233  
2  -30.084685  136.996784  

In [5]:
filepath = '../../data/weather_data/wettest_places.csv'
df_wettest_places=pd.read_csv(filepath)


df_wettest_places = df_wettest_places.drop(columns=["max_temperature", "min_temperature"]) #only interested in keeping max_rainfall
df_wettest_places = pd.merge(df_wettest_places, coordinates_df, on="region", how="left")
df_wettest_places.to_csv("../../data/weather_data/wettest_places.csv", index=False)

# Initial Exploration of Map Creation

## Choosing Between `geopandas` and `folium`

During the initial exploration, I considered using either `geopandas` or `folium` for mapping. After deliberation and reading about the features of both, I realized that `folium` was the better choice for creating **interactive** maps.  

We wanted to generate **dynamic** maps, which `geopandas` does not support. Instead, `geopandas` is more suited for static visualizations and geospatial data analysis.  

One key motivation for using `folium` was my use of **forecasted data**—I wanted to visualize how Pokémon move across the world as certain places get **warmer** and others get **cooler**. `folium`'s support for animations, interactive layers, and popups made it the ideal choice.

---

## Basic Functionality and Map Features

In the section below, I explore the basic functions of `folium`, which Adrian and Hailey further develop later in **[XXX]()**.  

The key steps include:  

1. **Converting Coordinates to Map Points**  
   - Extracting the latitude and longitude of the **hottest** and **coldest** places we identified.  
   - Plotting them on a map using `folium.Marker()`.  

2. **Adding Tooltips and Icons**  
   - Creating **tooltips** (hoverable text) to display location details.  
   - Using different **icons** to represent various points of interest.  

3. **Visualizing Temperature Ranges**  
   - Representing temperature variations using **color intensity**.  
   - Implementing a **heatmap** to illustrate how temperature changes dynamically over time.  

---

In [6]:
import folium
from branca.colormap import linear
import pandas as pd

# Initialise the base map centered between hottest and coldest places
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create color scales for hot and cold places
hot_colormap = linear.Reds_09.scale(df_hottest_places['temperature'].min(), df_hottest_places['temperature'].max())
cold_colormap = linear.Blues_09.scale(df_coldest_places['temperature'].min(), df_coldest_places['temperature'].max())

# Add markers for hottest places with gradient colors
for idx, row in df_hottest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=hot_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add markers for coldest places with gradient colors
for idx, row in df_coldest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=cold_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add legends to the map
hot_colormap.caption = 'Temperature Scale (Hottest Places)'
cold_colormap.caption = 'Temperature Scale (Coldest Places)'
hot_colormap.add_to(m)
cold_colormap.add_to(m)

# Save or display the map
m.save("temperature_ranked_map.html")
m


In [7]:
# Initialise the base map centered at an average location
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create a color scale for temperatures
colormap = linear.RdYlBu_11.scale(df_coldest_places['temperature'].min(), df_hottest_places['temperature'].max())

# Function to map temperature to color intensity
def get_marker_color(temp, temp_min, temp_max, color_scale):
    # Map temperature to a hex color
    hex_color = color_scale(temp)
    return hex_color

# Add markers for hottest places
for _, row in df_hottest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(m)

# Add markers for coldest places
for _, row in df_coldest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(m)

# Add color scale legend to map
colormap.caption = 'Temperature Scale (°C)'
colormap.add_to(m)

# Add HeatMap for temperature intensity
heat_data = [[row['latitude'], row['longitude'], row['temperature']] 
             for _, row in pd.concat([df_hottest_places, df_coldest_places]).iterrows()]
HeatMap(heat_data).add_to(m)

# Save or display the map
m.save("temperature_map_with_scale.html")
m


# Creation of Database!
* Note that pre-creation of database df.sort_values() function was used during collection of data from [openmeteo](https://open-meteo.com/), so data in [coldest_places.csv](coldest_places.csv), [hottest_places.csv](hottest_places.csv) and [wettest_places.csv](wettest_places.csv) is already ordered from highest rank to lowest rank, so all that is left is to rank them accordingly using an additional rankings column.

In [8]:
import sqlite3

# Load CSV files
coldest_places = pd.read_csv("../../data/weather_data/coldest_places.csv")
hottest_places = pd.read_csv("../../data/weather_data/hottest_places.csv")
wettest_places = pd.read_csv("../../data/weather_data/wettest_places.csv") 

# Sort coldest places and assign unique ranking
coldest_places = coldest_places.sort_values(by="temperature", ascending=True)
coldest_places["ranking"] = range(1, len(coldest_places) + 1) # range(1,101) is used for rankings to be generated from 1-100, quirok of range function:)

# Sort hottest places and assign unique ranking
hottest_places = hottest_places.sort_values(by="temperature", ascending=False)
hottest_places["ranking"] = range(1, len(hottest_places) + 1)

# Sort wettest places and assign unique ranking
wettest_places = wettest_places.sort_values(by="max_rainfall", ascending=False)
wettest_places["ranking"] = range(1, len(wettest_places) + 1)

# Connect to SQLite database
conn = sqlite3.connect("../../data/main.db")
cursor = conn.cursor()

# Drop existing tables if they exist
cursor.executescript("""
DROP TABLE IF EXISTS hottest_places;
DROP TABLE IF EXISTS coldest_places;
DROP TABLE IF EXISTS wettest_places;

CREATE TABLE hottest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    temperature FLOAT,
    latitude FLOAT,
    longitude FLOAT
);

CREATE TABLE coldest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    temperature FLOAT,
    latitude FLOAT,
    longitude FLOAT
);

CREATE TABLE wettest_places (
    ranking INTEGER PRIMARY KEY,
    region TEXT,
    max_rainfall FLOAT,
    latitude FLOAT,
    longitude FLOAT
);
""")

# Insert data into tables
coldest_places[["ranking", "region", "temperature", "latitude", "longitude"]].to_sql("coldest_places", conn, if_exists="append", index=False)
hottest_places[["ranking", "region", "temperature", "latitude", "longitude"]].to_sql("hottest_places", conn, if_exists="append", index=False)
wettest_places[["ranking", "region", "max_rainfall", "latitude", "longitude"]].to_sql("wettest_places", conn, if_exists="append", index=False)

# Commit and close connection
conn.commit()
conn.close()



## Exploration of Hottest and Coldest Places

From my folium exploration, I realized that most of the hottest places were located in the bottom half of the world. This sparked my curiosity to explore this trend further. I also wanted to analyse the variation in temperatures between the hottest and coldest places. Specifically, I was interested in whether there is a significant difference in temperature variance between the coldest and hottest locations.

The section below covers some basic visualizations based on my exploration. I used seaborn (which I picked up from python pre-sessional) and regression tools (which I have been slowly learning in 202W)
